## Overview

This analysis will expand on the implementation of one of most popular deep learning approach for time series modeling, LSTM. The approach has been introduced in the one of the lectures in Module 5 -- review the lecture for futher details. The analysis will be applied to the Bitcoin price data.

#### Setup

We set up first the analysis to be run under the main ISYE6402 repository Module 5. Then we import the libraries requried for this analysis.

In [ ]:
!pip install neuralforecast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.2/263.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.4/287.4 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.4/71.4 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.0
    Uninstalling click-8.3.0:
      Successfully uninstalled click-8.3.0


In [ ]:
# Install only if outside of the ISYE6402Main environment
#!pip install neuralforecast
! pip install pytorch_lightning

# Set the environment folder to be within the ISYE6402Main
import sys
import os
from pathlib import Path
# cwd = Path.cwd()
# initCwd = Path.cwd()
# while cwd != Path('/'):
#     if cwd.name == 'ISyE6402Main':
#         os.chdir(cwd)
#         os.chdir('Module5')
#         sys.path.insert(0, str(cwd))
#         break
#     cwd = cwd.parent
# if cwd.name != 'ISyE6402Main':
#     raise Exception(f"Unexpected working directory {initCwd}. Please check the correct directory.")

# Load Python Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from neuralforecast.models import LSTM
from neuralforecast.core import NeuralForecast
from neuralforecast.losses.pytorch import DistributionLoss, MAE, MSE
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error
)



#### Prediction Accuracy Measures

The function below is used to evaluate the prediction accuracy across multiple prediction error measures.

In [ ]:
# ========== evaluation methods ==========
def evaluate_performance(true, pred, model_name="Model"):
    mspe = mean_squared_error(true, pred)
    mae = mean_absolute_error(true, pred)
    mape = np.mean(np.abs((true - pred) / (true + 1e-6)))
    pm = (np.sum((true - pred) ** 2) / np.sum((true - np.mean(true)) ** 2))

    print(f"=== {model_name} Performance ===")
    print(f"MSPE: {mspe:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"MAPE: {mape:.4f}")
    print(f"PM:   {pm:.4f}")
    print("-" * 40)


#### Importing data

We import data from the 'Data' folder available with the Module 5 Jupyter notebooks.

In [ ]:

# ========== Load the raw BTC/USD historical data ==========
df = pd.read_csv('BTCUSD.csv')

# ========== Convert 'Date' column to datetime format ==========
df['Date'] = pd.to_datetime(df['Date'])

# ========== Filter data between 2019-04-01 and 2025-04-01 ==========
start_date = pd.to_datetime('2019-04-01')
end_date = pd.to_datetime('2025-04-01')
df = df[(df['Date'] >= start_date) & (df['Date'] <= end_date)]

# ========== Keep only 'Date' and 'Close' columns and rename them ==========
df = df[['Date', 'Close']].rename(columns={'Date': 'ds', 'Close': 'y'})

# ========== Add 'unique_id' column (required by NeuralForecast) ==========
df['unique_id'] = 'ts1'  # Single time series

# # ========== Rearrange columns and sort by date ==========
# df = df[['unique_id', 'ds', 'y']].sort_values('ds').reset_index(drop=True)

# ========== Preview the cleaned dataset ==========
print("First 5 rows:")
print(df.head())
print("\nLast 5 rows:")
print(df.tail())
# ========== Split the dataset into training and testing sets ==========
# Use all but the last 7 rows for training
train_data = df.iloc[:-7].reset_index(drop=True)

# Use the last 7 rows for testing
test_data = df.iloc[-7:].reset_index(drop=True)


First 5 rows:
             ds            y unique_id
1657 2019-04-01  4158.183105       ts1
1658 2019-04-02  4879.877930       ts1
1659 2019-04-03  4973.021973       ts1
1660 2019-04-04  4922.798828       ts1
1661 2019-04-05  5036.681152       ts1

Last 5 rows:
             ds             y unique_id
3845 2025-03-28  84353.148438       ts1
3846 2025-03-29  82597.585938       ts1
3847 2025-03-30  82334.523438       ts1
3848 2025-03-31  82548.914062       ts1
3849 2025-04-01  85169.171875       ts1


### Model Introduction

NeuralForecast is a deep learning library specifically designed for time series forecasting. It offers a simple interface and powerful functionality. Compared to traditional frameworks like PyTorch or TensorFlow, NeuralForecast is better suited for time series analysis because it comes with many built-in components tailored for forecasting tasks. These pre-configured features allow users to focus on model selection and performance evaluation without spending time on low-level implementation details

### Key Components





1.   h: int, forecast horizon.
2.   input_size: int, maximum sequence length for truncated train backpropagation. Default -1 uses 3 * horizon
3.  loss: training loss function.
4.  scaler_type: str=‘robust’, type of scaler for temporal inputs normalization
5.  encoder_n_layers, encoder_hidden_size: Number of LSTM layers and hidden state size.
6.  decoder_hidden_size, decoder_layers: MLP decoder architecture.
7.  max_steps: maximum number of training steps.



#### Hyperparameter Tuning: *input_size*

This analysis compares the prediction accuracy for using different input sizes.

In [ ]:
# ========== experiment setup ==========
horizon = 7
input_sizes = [-1, 30, 60, 90, 180,365]

for input_size in input_sizes:
    print(f"\nTraining LSTM with input_size = {input_size}")

    # Model setup
    lstm_model = LSTM(
        h=horizon,
        input_size=input_size,
        loss=DistributionLoss(distribution='Normal', level=[90, 95]),
        scaler_type='robust',
        encoder_n_layers=2,
        encoder_hidden_size=128,
        decoder_hidden_size=128,
        decoder_layers=2,
        max_steps=200,
        #trainer=trainer
    )

    nf = NeuralForecast(models=[lstm_model], freq='D')
    nf.fit(df=train_data)

    # Forecast and merge
    forecasts = nf.predict()
    merged = forecasts.merge(test_data, on='ds', how='inner')

    # Extract true and predicted
    y_true = merged['y'].values
    y_pred = merged['LSTM'].values

    evaluate_performance(y_true, y_pred, model_name=f"LSTM (input_size={input_size})")


/usr/local/lib/python3.12/dist-packages/neuralforecast/common/_base_model.py:151: UserWarning: Input size too small. Automatically setting input size to 3 * horizon = 21
  warnings.warn(
INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs



Training LSTM with input_size = -1


INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type             | Params | Mode 
----------------------------------------------------------
0 | loss         | DistributionLoss | 5      | train
1 | padder_train | ConstantPad1d    | 0      | train
2 | scaler       | TemporalNorm     | 0      | train
3 | hist_encoder | LSTM             | 199 K  | train
4 | mlp_decoder  | MLP              | 16.8 K | train
----------------------------------------------------------
215 K     Trainable params
5         Non-trainable params
215 K     Total params
0.864     Total estimated model params size (MB)
10        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=200` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Predicting: |          | 0/? [00:00<?, ?it/s]

=== LSTM (input_size=365) Performance ===
MSPE: 5727990.7774
MAE:  2061.3248
MAPE: 0.0243
PM:   1.5730
----------------------------------------


input_size = -1 yields the best model performance, which means that using the default setting of input_size = 3 × horizon = 21 provides the most optimal forecast accuracy among the current set of parameters.

#### Components Analysis: *scaler_type*

This analysis compares the prediction accuracy for using different approaches to scaling the data.

In [ ]:
# ========== experiment setup ==========
horizon = 7
scaler_types = ['identity', 'standard', 'robust']

for scaler in scaler_types:
    print(f"\nTraining LSTM with scaler_type = '{scaler}'")

    # Model setup
    lstm_model = LSTM(
        h=horizon,
        input_size=-1,
        loss=DistributionLoss(distribution='Normal', level=[90, 95]),
        scaler_type=scaler,
        encoder_n_layers=2,
        encoder_hidden_size=128,
        decoder_hidden_size=128,
        decoder_layers=2,
        max_steps=200,
        verbose=False,
        trainer=trainer
    )

    nf = NeuralForecast(models=[lstm_model], freq='D')
    nf.fit(df=train_data)

    # Forecast and merge
    forecasts = nf.predict()
    merged = forecasts.merge(test_data, on='ds', how='inner')

    # Extract true and predicted
    y_true = merged['y'].values
    y_pred = merged['LSTM'].values

    # Evaluate
    evaluate_performance(y_true, y_pred, model_name=f"LSTM (scaler_type='{scaler}')")


Robust Scaler uses a normalization approach based on the median and a robust measure of scale, allowing the model to train and predict stably even in the presence of outliers. Therefore, compared to other methods such as 'identity' and 'standard', Robust offers clear advantages.

#### Components Analysis: Model Structure (encoder_n_layers & hindden_size)

This analysis compares the prediction accuracy for using different inputs for the number of layers in the enconder and hidden levels.

In [ ]:
# ========== experiment setup ==========
horizon = 7
input_size = -1
scaler_type = 'robust'

structure_combinations = [
    (1, 64),
    (1, 128),
    (2, 128),   # baseline
    (2, 256),
    (3, 256),
    (3, 512)
]

for n_layers, hidden_size in structure_combinations:
    print(f"\nTraining LSTM with encoder_n_layers = {n_layers}, encoder_hidden_size = {hidden_size}")

    # Model setup
    lstm_model = LSTM(
        h=horizon,
        input_size=input_size,
        loss=DistributionLoss(distribution='Normal', level=[90, 95]),
        scaler_type=scaler_type,
        encoder_n_layers=n_layers,
        encoder_hidden_size=hidden_size,
        decoder_hidden_size=hidden_size,
        decoder_layers=2,
        max_steps=200,
        verbose=False,
        trainer=trainer
    )

    nf = NeuralForecast(models=[lstm_model], freq='D')
    nf.fit(df=train_data)

    # Forecast and merge
    forecasts = nf.predict()
    merged = forecasts.merge(test_data, on='ds', how='inner')

    # Extract true and predicted
    y_true = merged['y'].values
    y_pred = merged['LSTM'].values

    # Evaluate
    evaluate_performance(y_true, y_pred, model_name=f"LSTM (layers={n_layers}, hidden={hidden_size})")


Among all tested configurations, the combination of 3 encoder layers and 512 hidden units yielded the best forecasting results across all evaluation metric.This suggests that the target time series exhibits complex temporal dependencies that benefit from deeper and wider LSTM networks. A deeper encoder allows the model to capture higher-level temporal patterns, while a larger hidden state improves memory capacity.

#### Analysis of Error Loss Function used in Model Learning

In our experiments, we compared three common loss functions for LSTM-based forecasting:

1. MAE and MSE focus on point accuracy and differ mainly in their sensitivity to large errors.

2. DistributionLoss adds uncertainty modeling, allowing the forecast to adapt its variance over time and provide prediction intervals，useful when quantifying risk is important.


In [ ]:
# ======== Loss candidates ========
loss_candidates = [
    ("Normal", DistributionLoss(distribution='Normal', level=[90, 95])),
    ("MAE", MAE()),
    ("MSE", MSE()),
]

results = []

for loss_name, loss_obj in loss_candidates:
    alias = f"LSTM_{loss_name}"
    print(f"\n=== Training {alias} ===")
    lstm_model = LSTM(
        h=7,
        input_size=-1,
        loss=loss_obj,
        scaler_type='robust',
        encoder_hidden_size=512,
        decoder_hidden_size=512,
        encoder_n_layers=3,
        decoder_layers=2,
        max_steps=200,
        alias=alias,
        verbose=False,
        trainer=trainer
    )

    # Fit & Predict
    nf = NeuralForecast(models=[lstm_model], freq='D')
    nf.fit(df=train_data)

    forecasts = nf.predict()

    merged = forecasts.merge(test_data[['unique_id', 'ds', 'y']], on=['unique_id', 'ds'], how='inner')


    y_true = merged['y'].values
    y_pred = merged[alias].values


    evaluate_performance(y_true, y_pred, model_name=f"{alias}")



From our results, DistributionLoss with a Normal assumption achieved the best performance, and was particularly helpful for highly volatile series such as BTC, where modeling time-varying uncertainty significantly improves forecast robustness.

### Final Tuned LSTM model

In [ ]:

horizon = 7
lstm_model = LSTM(
    h=horizon,
    input_size=-1,
    #context_size=30,
    loss=DistributionLoss(distribution='Normal', level=[90, 95]),
    #loss=MAE(),
    scaler_type='robust',
    encoder_n_layers=3,
    encoder_hidden_size=512,
    decoder_hidden_size=512,
    decoder_layers=2,
    max_steps=200,
    verbose=False,
    trainer=trainer
)

nf = NeuralForecast(models=[lstm_model],freq='D')
nf.fit(df=train_data)


In [ ]:
# Predict
forecasts = nf.predict()
# Merge forecast with test set for comparison
merged = forecasts.merge(test_data, on='ds', how='inner')
plt.figure(figsize=(12, 6))


# Get the start date of the test set
test_start_date = test_data['ds'].min()
# Select last 45 days from train_data before test starts
train_subset = train_data[train_data['ds'] >= test_start_date - pd.Timedelta(days=45)]
# Plot training data
plt.plot(train_subset['ds'], train_subset['y'], label='Train (last 45 days)', color='black')

# Plot test data (ground truth)
plt.plot(merged['ds'], merged['y'], label='Test (Ground Truth)', color='green')

# Plot forecast mean (LSTM prediction)
if 'LSTM' in merged.columns:
    plt.plot(merged['ds'], merged['LSTM'], label='LSTM Forecast', color='red', linestyle='--')
# Plot 90% confidence interval
if 'LSTM-lo-90' in merged.columns and 'LSTM-hi-90' in merged.columns:
    plt.fill_between(merged['ds'], merged['LSTM-lo-90'], merged['LSTM-hi-90'],
                     alpha=0.15, color='blue', label='90% Confidence Interval')

# Plot 95% confidence interval
if 'LSTM-lo-95' in merged.columns and 'LSTM-hi-95' in merged.columns:
    plt.fill_between(merged['ds'], merged['LSTM-lo-95'], merged['LSTM-hi-95'],
                     alpha=0.3, color='orange', label='95% Confidence Interval')

# Final plot adjustments
plt.title('Tuned LSTM Forecast')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()



In [ ]:
print("Train last date:", train_data['ds'].max())
print("Forecast start:", forecasts['ds'].min())
print("Test start:", test_data['ds'].min())


In [ ]:
# Align forecast and test set
aligned = forecasts.merge(test_data, on='ds', how='inner')
# Get values
actual = aligned['y']
predicted = aligned['LSTM']
# Evaluate
evaluate_performance(actual, predicted, model_name="LSTM")

=== LSTM Performance ===
MSPE: 3542882.6780
MAE:  1617.2411
MAPE: 0.0191
PM:   0.9729
----------------------------------------


### Adding Exogeneous Variables

In this LSTM setup, `hist_exog_list` marks **historical exogenous features** (e.g., lagged prices for SP500 and Gold). At every step,  the most recent `input_size` time steps are input to the model, feeding the full window of those exogenous variables alongside the target series. Effectively, this provides multi-lag information (lags 1 … `input_size`) automatically, with no manual shifting needed and no future values required during prediction, as long as the latest `input_size` days of data are present.


In [ ]:
# ========= Step 1: Load & Prepare Data =========
df = pd.read_csv('merged_data.csv')
df['Date'] = pd.to_datetime(df['Date'])
# Keep data between 2019-04-01 and 2025-04-01 (inclusive)
df = df[(df['Date'] >= '2019-04-01') & (df['Date'] <= '2025-04-01')]
# Rename columns to fit NeuralForecast conventions
df = df.rename(columns={'Date': 'ds'})
df['unique_id'] = 'ts1'
df = df[['unique_id', 'ds', 'y', 'SP500', 'Gold']].sort_values('ds')

# ========= Step 2: Train / Test Split =========
train_data = df.iloc[:-7].reset_index(drop=True)
test_data  = df.iloc[-7:].reset_index(drop=True)

# ========= Step 3: Define & Train Model =========
lstm_model_R = LSTM(
    h=7,                           # forecast horizon = 7 days
    input_size=-1,                 # default = 3 * h = 21 time steps look-back
    loss=DistributionLoss('Normal', level=[90, 95]),
    scaler_type='robust',
    encoder_n_layers=3,
    encoder_hidden_size=512,
    decoder_hidden_size=512,
    decoder_layers=2,
    max_steps=200,
    hist_exog_list=['SP500', 'Gold'],   # treat SP500 & Gold as historical exogenous
     verbose=False,
    trainer=trainer
)

nf = NeuralForecast(models=[lstm_model_R], freq='D')
nf.fit(df=train_data)

# ========= Step 4: Predict =========
forecasts = nf.predict()

# ========= Step 5: Merge & Plot =========
merged = forecasts.merge(test_data[['ds', 'y']], on='ds', how='inner')

plt.figure(figsize=(12, 6))

# Provide 45 days of recent training data for visual context
ctx_start = test_data['ds'].min() - pd.Timedelta(days=45)
train_ctx = train_data[train_data['ds'] >= ctx_start]

plt.plot(train_ctx['ds'], train_ctx['y'],
         label='Train (last 45 days)', color='black')

plt.plot(merged['ds'], merged['y'],
         label='Test (Ground Truth)', color='green')

plt.plot(merged['ds'], merged['LSTM'],
         label='LSTM Forecast', color='red', linestyle='--')

# Plot prediction intervals
plt.fill_between(merged['ds'], merged['LSTM-lo-90'], merged['LSTM-hi-90'],
                 alpha=0.15, color='blue', label='90% CI')
plt.fill_between(merged['ds'], merged['LSTM-lo-95'], merged['LSTM-hi-95'],
                 alpha=0.30, color='orange', label='95% CI')

plt.title('LSTM Forecast with Historical Exogenous Variables (SP500 & Gold)')
plt.xlabel('Date')
plt.ylabel('y')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Align forecast and test set
aligned = forecasts.merge(test_data, on='ds', how='inner')

# Get values
actual = aligned['y']
predicted = aligned['LSTM']

# Evaluate
evaluate_performance(actual, predicted, model_name="LSTM_R")

=== LSTM_R Performance ===
MSPE: 4388885.2148
MAE:  1898.1752
MAPE: 0.0224
PM:   1.2052
----------------------------------------
